# Czysty notatnik do generowania działającego (miejmy nadzieję) modelu detekcji zagrożeń na podstawie logów Wazuh
## ✨Organicznie uprawiany kod✨
Zalecam korzystanie z menedżera środowisk python `uv`, bo jest lepszy i nie tak ociężały jak `conda` czy domyślny `venv`.
Tworzenie środowiska
```
uv venv .venv --python cp3.14 # argument --python opcjonalnie cp3.14 po prostu jest szybki ;)
```
Aktywacja środowiska
```
source .venv/bin/activate     # to pewnie będzie inaczej na Windows
```
Instalacja bibliotek
```
uv pip install ipykernel pandas numpy
```

In [149]:
import re
# czy wykonywać kod upraszczający analizę danych
TESTY = False
DATA_DIR = "..//data//ait_ads"
TOKEN_RX = re.compile(r"[A-Za-z0-9+/=%_.-]{8,}")
REQ_RX = re.compile(r'"([A-Z]+)\s+(\S+)\s+HTTP/[0-9.]"\s+(\d{3})')
URL_FLAGS = {
    "urlflag_wpadmin": "wp-admin",
    "urlflag_wpcontent": "wp-content",
    "urlflag_wpjson": "wp-json",
    "urlflag_wplogin": "wp-login",
    "urlflag_xmlrpc": "xmlrpc",
    "urlflag_wpconfig": "wp-config",
    "urlflag_phpmyadmin": "phpmyadmin",
    "urlflag_backup": "backup",
    "urlflag_dump": "dump",
    "urlflag_config": "config",
    "urlflag_git": ".git",
    "urlflag_sql": "sql",
    "urlflag_readme": "readme",
    "urlflag_license": "license",
    "urlflag_test": "test",
    "urlflag_dev": "dev",
    "urlflag_secret": "secret",
    "urlflag_admin": "admin",
    "urlflag_pass": "pass",
    "urlflag_user": "user",
    "urlflag_include": "include",
    "urlflag_proc": "proc",
    "urlflag_ocs": "ocs/",
    "urlflag_remote": "remote",
    "urlflag_shell": "shell",
    "urlflag_cmd": "cmd",
    "urlflag_php": ".php",
    "urlflag_txt": ".txt",
    "urlflag_zip": ".zip",
}
UA_PATTERNS = {
    "ua_dirb": "dirb",
    "ua_wpscan": "wpscan",
    "ua_nmap": "nmap",
    "ua_ncat": "ncat",
    "ua_wget": "wget",
    "ua_headless": "HeadlessChrome",
}
SCENARIOS = ["fox", "harrison", "russellmitchell", "santos", "shaw", "wardbeck", "wheeler", "wilson"]


In [150]:
import json

records = []
# Load from a file
with open("../data/ait_ads/fox_wazuh.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:  # skip blank lines
            record = json.loads(line)
            if not "Suricata" in record["rule"]["description"] and not "suricata" in record["rule"]["groups"]:
                records.append(record)

if TESTY:
    print(records[0])

In [151]:
import pandas as pd
df = pd.json_normalize(records)
records.clear()

if TESTY:
    print(df.shape)
    df.head()

In [152]:
def print_unique_values(df):
    for col in df.columns:
        try:
            counts = df[col].value_counts(dropna=False)
        except TypeError:  # lists or dicts aren't hashable
            counts = df[col].astype(str).value_counts(dropna=False)

        print(f"\n=== {col} ({counts.shape[0]} unique) ===")
        print(counts)

if TESTY:
    print_unique_values(df)

**Your feature set has a likely leakage problem: several of the `rule.*` fields are derived from the same rules that produced the label, so the model may learn the rules rather than the attacks.** Before training, check how `attack` was labeled. If it came from Wazuh rule levels, groups, or MITRE mappings, those features will make the task trivially easy and the model won't generalize to new traffic.

Here's how I'd group your features:

| Group | Features | Recommendation |
|---|---|---|
| **Likely leakage** | `rule.level`, `rule.groups`, `rule.mitre.*`, `rule.pci_dss`, `rule.tsc`, `rule.nist_800_53`, `rule.gpg13`, `rule.gdpr`, `rule.hipaa` | Drop these if the label depends on Wazuh's rules. The compliance fields are just rule metadata, so they mostly identify the rule that fired. |
| **Identifiers** | `id`, `data.id`, `full_log` (as raw text) | Drop `id` since it's unique per alert and carries no signal. `data.id` is probably a user or session identifier and risks memorization. `full_log` needs parsing or vectorization, not direct use. |
| **Time** | `@timestamp` | Don't feed raw timestamps. Extract hour of day, day of week, and possibly inter-event gaps per source IP. |
| **Network** | `data.srcip`, `data.dstip`, `data.srcport`, `data.protocol`, `data.url` | Keep these but encode carefully. IPs are high cardinality, so use derived features like counts per IP over a time window, whether the IP is internal or external, or port categories. |
| **Identity and session** | `data.srcuser`, `data.dstuser`, `data.tty` | Usually sparse. Consider flags such as "login as root" or "failed attempts per user" instead of raw values. |
| **Geo** | `GeoLocation.*` | Mostly populated only for external IPs. Consider a missing-value indicator, and beware that country-level features can encode the data source more than attack behavior. |
| **Source metadata** | `location`, `predecoder.*`, `decoder.name`, `decoder.parent` | These describe which log produced the event. They're useful for context but can cause the model to learn log-source imbalances. |
| **Unclear** | `previous_output` | I can't tell what this is from the name. If it's the output of an earlier pipeline stage or a prior model, check that it isn't built from the label. |

Two other things to check. First, time-based splits matter more than random splits here, since attack patterns drift and random splits can inflate your scores. Second, sparse fields like `data.url` and the `GeoLocation` fields may be mostly null, so confirm their null rates before relying on them.

What does the `attack` label come from, and what's `previous_output`? With those answers I can say more precisely which features to keep.

In [153]:
df = df.drop(columns=[
    "agent.name", "agent.id", # id, agent.ip drop później
    "manager.name",
    "rule.firedtimes", "rule.mail", "rule.description", "rule.id", "rule.frequency", "rule.level", "rule.groups", "rule.mitre.technique", "rule.mitre.id", "rule.mitre.tactic", "rule.pci_dss", "rule.tsc", "rule.nist_800_53", "rule.gpg13", "rule.gdpr", "rule.hipaa", # zastąpić przez fired times ale w oknie
    "predecoder.timestamp",
    "input.type",
    "data.audit.uid", "data.audit.auid", "data.audit.gid", "data.audit.exe", "data.audit.euid", "data.audit.session", "data.audit.pid", "data.audit.id", "data.audit.type", "data.audit.command", "data.uid", "data.pwd", "data.command", "data.euid", "data.id", # wyglądają bezurzytecznie
    "decoder.ftscomment", "decoder.name", "decoder.parent",
    "GeoLocation.city_name", "GeoLocation.country_name", "GeoLocation.location.lat", "GeoLocation.location.lon", "GeoLocation.region_name", "location",
    "predecoder.hostname", 'predecoder.program_name',
    "previous_output"
], errors="ignore")

# jeżeli location = "/var/log/apache2/intranet-access.log" to host_role = WWW
# data.srcuser zamienić na is_known_user


In [154]:
if TESTY:
    print_unique_values(df)
    print(df.shape)
    print(df.columns)

In [155]:
def port_category(port):
    if port == 80:
        return "http"
    elif port == 53:
        return "dns"
    elif port == 443:
        return "https"
    elif port == 587:
        return "smtp"
    elif port == 5800:
        return "vnc"
    else:
        return "other"

if TESTY:
    counts = df['data.dstip'].value_counts(dropna=False)
    print(counts.keys())
    ports = []
    for key in counts.keys()[1:]:
        split = key.split(':')
        if len(split) == 2 and not split[1] in ports:
            ports.append(split[1])
    print(ports)
    print(len(ports))

In [156]:
local_ips = df['agent.ip'].value_counts(dropna=False).keys()[1:]

def is_ip_internal(ip):
    if type(ip) != str:
        return False
    split = ip.split(':')
    #local_ips = df['agent.ip'].value_counts(dropna=False).keys()[1:]
    if split[0] in local_ips:
        return True
    else:
        return False


In [157]:
from datetime import UTC, datetime
from collections import Counter
import math

def parse_iso(value: object) -> float | None:
    if not isinstance(value, str) or not value:
        return None
    text = value.rstrip("Z")
    for fmt in ("%Y-%m-%dT%H:%M:%S.%f%z", "%Y-%m-%dT%H:%M:%S%z", "%Y-%m-%dT%H:%M:%S.%f", "%Y-%m-%dT%H:%M:%S"):
        try:
            parsed = datetime.strptime(text, fmt)
        except ValueError:
            continue
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=UTC)
        return parsed.timestamp()
    return None

def hour(ts):
    iso_ts = parse_iso(ts)
    stamp = datetime.fromtimestamp(iso_ts, tz=UTC)
    # interval = ts # nwm to trzeba zrobić
    return float(stamp.hour)

def day(ts):
    iso_ts = parse_iso(ts)
    stamp = datetime.fromtimestamp(iso_ts, tz=UTC)
    # interval = ts # nwm to trzeba zrobić
    return float(stamp.weekday())

def entropy_of(text: str) -> float:
    if type(text) != str:
        return 0.0
    counts = Counter(text)
    total = len(text)
    value = 0.0
    for count in counts.values():
        probability = count / total
        value -= probability * math.log2(probability)
    return round(value, 4)

In [158]:
import numpy as np
df["unix_time"] = df["@timestamp"].apply(lambda x: parse_iso(x))

df["is_dst_user_root"] = np.where(df["data.dstuser"] == "root", 1, 0)
# data.srcip -> ip wew lub zew
df["is_srcip_internal"] = df["data.srcip"].apply(is_ip_internal)
# data.dstip -> ip wew lub zew
df["is_dstip_internal"] = df["data.dstip"].apply(is_ip_internal)
# full_log -> wektoryzacja

# @timestamp -> godzina_dnia, dzień_tygodnia, czas_między_ostatnim_tym_samym_srcip
df["hour"] = df["@timestamp"].apply(hour)
df["weekday"] = df["@timestamp"].apply(day)
# kategoria portu
df["dstport"] = df["data.dstip"].apply(port_category)
# cechy url
df["url_entropy"] = df["data.url"].apply(entropy_of)
df["url_len"] = df["data.url"].apply(lambda x: len(x) if type(x) == str else 0 )
df["url_parts"] = df["data.url"].apply(lambda x: len(x.split("/")) if type(x) is str else 0)
df["url_query"] = df["data.url"].apply(lambda x: "?" in x if type(x) is str else 0)
## wymagane do needle scanu, po wszystkim url i tak trzeba usunąć
df["data.url"] = df["data.url"].apply(lambda x: x.lower() if type(x) is str else 0)
## needle scan
for flag, needle in URL_FLAGS.items():
    df[flag] = df["data.url"].apply(lambda x: float(needle in x if type(x) is str else False))
for flag, needle in UA_PATTERNS.items():
    df[flag] = df["data.url"].apply(lambda x: float(needle.lower() in x if type(x) is str else False))
df["url_tokens"] = df["data.url"].apply(lambda x: float(len(re.findall(r"[A-Za-z0-9]+", x)) if type(x) is str else 0))
df["url_encoded_chars"] = df["data.url"].apply(lambda x: float(x.count("%") if type(x) is str else 0))
df["url_max_token"] = df["data.url"].apply(lambda x: float(max((len(t) for t in TOKEN_RX.findall(x)), default=0)) if type(x) is str else 0)
# ograniczenie full_log
df["log_head"] = df["full_log"].apply(lambda x: re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', '', x[:400]) if type(x) is str else "")

df["scenario"] = "fox"

In [159]:
# oznaczanie ataków
import csv

def load_phases(path: str) -> pd.DataFrame:
    rows = []
    with open(path, newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            rows.append({
                'scenario':    row["scenario"].strip(),
                'attack_type': row["attack"].strip(),
                'start':       float(row["start"].strip()),
                'end':         float(row["end"].strip()),
            })
    return pd.DataFrame(rows, columns=['scenario', 'attack_type', 'start', 'end'])

phases = load_phases("../data/ait_ads/labels.csv")

def classify_vectorized(df, phases):
    labels = np.full(len(df), "background", dtype=object)
    types  = np.full(len(df), "", dtype=object)
    for _, phase in phases.iterrows():
        mask = (df["unix_time"] >= phase["start"]) & (df["unix_time"] < phase["end"])
        labels[mask] = "attack"
        types[mask]  = phase["attack_type"]
    return pd.DataFrame({"attack": labels, "attack_type": types}, index=df.index)

df = pd.concat([df, classify_vectorized(df, phases)], axis=1)

In [160]:
# Drop zbędnych kolumn
df = df.drop(columns=["full_log", "agent.ip", "data.url", "data.srcip", "data.dstip", "data.dstuser", "data.srcuser"], errors="ignore")

if TESTY:
    print_unique_values(df)
    df.shape

In [161]:
#def url_features(url: object, scan_text: str) -> dict[str, float]:
#    text = as_str(url)
#    lowered = text.lower()
#    out: dict[str, float] = {
#        "url_len": float(len(text)),
#        "url_parts": float(len([part for part in lowered.split("/") if part])),
#        "url_query": float("?" in text),
#        "url_tokens": float(len(re.findall(r"[A-Za-z0-9]+", lowered))),
#        "url_encoded_chars": float(lowered.count("%")),
#        "url_entropy": entropy_of(text),
#        "url_max_token_len": float(max((len(t) for t in TOKEN_RX.findall(text)), default=0)),
#    }
#    for flag, needle in URL_FLAGS.items():
#        out[flag] = float(needle in lowered)
#    low_scan = (scan_text or "").lower()  # !!! to nie wiem co tu robi, ale chyba nie jest "krytyczne"
#    for flag, needle in UA_PATTERNS.items():
        # needles are case-insensitive here; low_scan is already lowercased
#        out[flag] = float(needle.lower() in low_scan)
#    return out

# może się przydać, ale nie teraz
def scan_text_for_url(raw: str) -> tuple[str, str, str]:
    match = REQ_RX.search(raw)
    if match:
        return match.group(2), match.group(1), match.group(3)
    return "", "", ""

# może się przydać, ale nie teraz
def log_resource_class(resources: list) -> str:
    text = " ".join(str(r) for r in resources).lower()
    if "access" in text or "apache" in text:
        return "apache"
    if "metricbeat" in text or "filebeat" in text:
        return "metrics"
    if "syslog" in text or "auth" in text:
        return "syslog"
    return "other"

# Przetwarzanie wewnątrz okna (szablon - nie brać na poważnie)

In [162]:
# firedtimes wewnątrz okna